# 🧮 Aula 02 — Modelos de Processamento: SIMD, MIMD, RISC e CISC

**Objetivo:** reconhecer os modelos de processamento paralelo (**SIMD** e **MIMD**) e
instrucional (**RISC** e **CISC**) — e usar isso para escolher hardware de IA embarcada.

**Roteiro:**
1. Detecção do backend de processamento (GPU ou CPU SIMD).
2. Taxonomia de Flynn: SIMD e MIMD.
3. Sequencial vs. SIMD com NumPy (mede o ganho).
4. RISC vs. CISC na prática (arquitetura da máquina).
5. Estudo de caso: imagem 1080p — SIMD + MIMD numa GPU.
6. Discussão e síntese.

> 💡 Roda no **Colab** (com GPU) e também no **Windows com GPU AMD** — o código detecta
> o melhor backend e continua mesmo sem GPU.

## 1. Detecção do backend

O mesmo script serve para Colab (CuPy/T4), Linux com ROCm e Windows com AMD. Ele detecta
a melhor engine disponível; sem GPU, usa o **SIMD da própria CPU** (AVX/SSE).

In [ ]:
# @title 🔍 Detectar o backend de processamento
# ============================================================================
# OBJETIVO: escolher a melhor "engine" de paralelismo disponível.
# Ordem: CuPy (GPU) > PyTorch CUDA > PyTorch DirectML (AMD/Win) > NumPy (CPU).
# ============================================================================
import os, sys
try:
    sys.stdout.reconfigure(encoding="utf-8")   # evita erro de acento no Windows
except Exception:
    pass

# Garante que a pasta scripts/ esteja no caminho de importação
# (no Colab, o notebook e a pasta scripts/ ficam lado a lado).
if os.path.isdir("scripts"):
    sys.path.insert(0, "scripts")

try:
    import lib_backend
except Exception:
    # Fallback: se a pasta scripts/ não estiver disponível, usa só NumPy.
    import types, numpy as _np
    lib_backend = types.SimpleNamespace()
    lib_backend.detectar_backend = lambda: _np
    lib_backend.info = lambda: {"backend": "NumPy (CPU SIMD)", "dispositivo": "CPU"}

# detectar_backend() escolhe a engine e devolve xp (numpy OU cupy)
xp = lib_backend.detectar_backend()
estado = lib_backend.info()
BACKEND, GPU_NOME = estado["backend"], estado["dispositivo"]
print("Backend:", BACKEND, "| Dispositivo:", GPU_NOME)


## 2. Taxonomia de Flynn

| Modelo | Instruções | Dados | Exemplo |
| :--- | :--- | :--- | :--- |
| **SISD** | 1 | 1 | CPU clássica sequencial |
| **SIMD** | 1 | Múltiplos | GPU, AVX/SSE, NumPy |
| **MISD** | Múltiplas | 1 | Raro (pipelines especializados) |
| **MIMD** | Múltiplas | Múltiplos | CPU multi-core, clusters |

Vamos medir SIMD (vetorizado) contra SISD (sequencial).

## 3. Demonstração: sequencial vs. SIMD (NumPy)

Somamos **1 milhão** de elementos de duas formas e medimos o tempo.

In [ ]:
# @title ⏱️ Sequencial (SISD) vs. vetorizado (SIMD)
# ============================================================================
# OBJETIVO: mostrar que "uma instrução para muitos dados" (SIMD) é muito mais
# rápido que processar um elemento por vez (SISD).
# ============================================================================
import time
import numpy as np

N = 1_000_000

# ── Versão sequencial: um elemento por vez (modelo SISD) ────────────────────
a = list(range(N)); b = list(range(N))
inicio = time.time()
c_seq = [a[i] + b[i] for i in range(N)]
t_seq = time.time() - inicio
print(f"Sequencial (1 por vez): {t_seq:.4f}s")

# ── Versão SIMD com NumPy: todos os elementos de uma vez ────────────────────
a_np = np.arange(N); b_np = np.arange(N)
inicio = time.time()
c_np = a_np + b_np          # o NumPy usa instruções SIMD (AVX/SSE) da CPU
t_np = time.time() - inicio
print(f"NumPy (SIMD):           {t_np:.6f}s")
print(f"-> Speedup: {t_seq / t_np:,.0f}x mais rápido\n")

# ── Validação ────────────────────────────────────────────────────────────────
assert list(c_seq) == c_np.tolist(), "Resultados divergem!"
print("[OK] Resultados conferem: sequencial == SIMD.")

### Se houver GPU

No Colab com T4, o mesmo código pode rodar na placa via CuPy:

In [ ]:
# @title 🚀 (Opcional) Mesma soma na GPU, se houver
# ============================================================================
# Só executa se o backend detectado for uma GPU. O código é quase idêntico:
# troca-se numpy por cupy — o "espírito" do SIMD é o mesmo.
# ============================================================================
import time
if BACKEND != "NumPy (CPU SIMD)":
    xa = xp.arange(N, dtype=np.float32) if "cupy" in BACKEND.lower() else xp.arange(N)
    xb = xp.arange(N, dtype=np.float32) if "cupy" in BACKEND.lower() else xp.arange(N)
    inicio = time.time(); _ = xa + xb
    print(f"GPU ({GPU_NOME}): {time.time() - inicio:.6f}s")
else:
    print("Sem GPU neste ambiente — a comparação CPU sequencial vs. NumPy/SIMD já basta.")

## 4. RISC vs. CISC na prática

A arquitetura da máquina aparece em `platform.machine()`: **x86/x64 = CISC**;
**ARM/aarch64 = RISC**. No Colab (Linux x86) dá para inspecionar também o assembly.

In [ ]:
# @title 🧩 Tipo de arquitetura e recursos SIMD da CPU
# ============================================================================
# OBJETIVO: identificar se a máquina é RISC ou CISC e quais extensões SIMD usa.
# ============================================================================
import platform

maquina = platform.machine()
if any(t in maquina.lower() for t in ("x86", "amd64")):
    tipo = "CISC (x86/x64) — instruções de tamanho VARIÁVEL"
elif "arm" in maquina.lower() or "aarch64" in maquina.lower():
    tipo = "RISC (ARM) — instruções de tamanho FIXO"
else:
    tipo = "Indefinida"
print(f"Máquina: {maquina}  ->  {tipo}")

# No Colab/Linux, também é possível ver o assembly (descomente no terminal):
# !cat /proc/cpuinfo | grep "model name" | head -1
# !objdump -d /bin/ls | head -30

## 5. Estudo de caso: imagem 1080p (SIMD + MIMD)

Converter uma imagem para tons de cinza mostra por que a GPU é feita para visão
computacional. Dentro do warp é **SIMD**; entre blocos/SMs é **MIMD**.

In [ ]:
# @title 🖼️ Tons de cinza: loop por pixel vs. vetorizado
# ============================================================================
# OBJETIVO: comparar o processamento sequencial (pixel a pixel) com o SIMD
# (todos os pixels de uma vez), como uma GPU faria.
# ============================================================================
import time
import numpy as np

H, W = 1080, 1920
rng = np.random.default_rng(42)             # semente fixa: reprodutível
imagem = rng.integers(0, 256, (H, W, 3), dtype=np.uint8)
print(f"Imagem {W}x{H} ({H * W:,} pixels)\n")

# ── CPU, pixel a pixel ──────────────────────────────────────────────────────
inicio = time.time()
cinza_loop = np.zeros((H, W), dtype=np.uint8)
for i in range(H):
    for j in range(W):
        r, g, b = imagem[i, j]
        cinza_loop[i, j] = int(0.299 * r + 0.587 * g + 0.114 * b)
t_loop = time.time() - inicio
print(f"CPU (loop):      {t_loop:.3f}s")

# ── Vetorizado (SIMD): todos os pixels de uma vez ───────────────────────────
inicio = time.time()
r = imagem[:, :, 0].astype(np.float32)
g = imagem[:, :, 1].astype(np.float32)
b = imagem[:, :, 2].astype(np.float32)
cinza_simd = (0.299 * r + 0.587 * g + 0.114 * b).astype(np.uint8)
t_simd = time.time() - inicio
print(f"SIMD/Vetorizado: {t_simd:.3f}s")
print(f"-> Speedup: {t_loop / t_simd:,.0f}x mais rápido!")

## 6. Discussão em Grupo

1. Raspberry Pi (ARM/RISC) ou NVIDIA Jetson (ARM + GPU) para visão computacional em tempo real?
2. Em que situações o CISC (x86) ainda vence o RISC (ARM)?
3. Um smartphone usa GPU para games. Por que ela também serve para reconhecimento facial?
4. Se SIMD acelera tanto, por que não colocamos SIMD em tudo?

> Atividade completa em `aulas/aula02/atividade.md`.

## 7. Síntese e Tarefa de Casa

- **SIMD** — 1 instrução → múltiplos dados (GPU, NumPy, AVX).
- **MIMD** — múltiplas instruções → múltiplos dados (multi-core, clusters).
- **RISC (ARM)** — instruções simples e fixas → baixo consumo → embarcados/mobile.
- **CISC (x86)** — instruções complexas e variáveis → desktops/servidores.
- **GPU** — SIMD dentro do warp + MIMD entre blocos.

**Tarefa (opcional):** compare **Jetson Nano** × **Raspberry Pi 4** (núcleos de CPU/GPU,
suporte a SIMD/CUDA, consumo em watts, preço e disponibilidade).

> 🔗 **Próxima aula:** *Estrutura de Memória em GPUs* — onde os dados ficam e por que o
> barramento é o próximo gargalo.